# FPGA implementation

This notebook describes the FPGA arithmetic kernels and reproduces their HLS timing estimates. Each system size is synthesized as a dedicated hardware datapath, exposing parallelism within an energy calculation. These are arithmetic-kernel estimates: proposal generation, random-number generation, the random acceptance comparison, and configuration and cached-energy updates belong to the caller.

**Table of contents**

1. [Source](#source)
2. [Algorithm](#algorithm)
   1. [Local move](#local-move)
   2. [Uniform move](#uniform-move)
3. [Reproducibility](#reproducibility)
4. [Timing calculation](#timing-calculation)

<a id="source"></a>
## Source

`local_move_hls.cpp` and `uniform_move_hls.cpp` implement the two HLS top-level functions. `fpga.hpp` defines the fixed-point types, energy arithmetic, and acceptance approximation. `wallace.hpp` implements a carry-save Wallace reduction and its final parallel-prefix adder. `make_exp_cheby.py` generates `fixed_config.hpp` and `exp_cheby.hpp`, containing the precision settings, cutoff, and polynomial coefficients.

`run_hls.tcl` creates a Vitis HLS component, selects the local or uniform kernel and its testbench, sets the device and target clock, and runs either C simulation or synthesis. The stage, proposal type, and system size are supplied through environment variables by `run.sh`.

<a id="algorithm"></a>
## Algorithm

The inputs are the physical Hamiltonian coefficients rounded to $b$ fractional bits, with no additional energy rescaling. Both kernels use the convention $0\mapsto-1$ and $1\mapsto+1$ for spin bits. The caller must make the coefficients available in parallel, for example from on-chip registers.

The generator sets $b=\lceil\log_2(\beta_{\rm ref}M/\ln(1+2\varepsilon_b))\rceil$, where $M=n(n+1)/2$, $\beta_{\rm ref}=1$, and $\varepsilon_b=10^{-4}$. The signed energy type has $I_n=1+\lceil\log_2(B_n+1)\rceil$ integer bits, including the sign, with $B_n=2\lceil n^{3/2}/\sqrt{\pi}\rceil$, and total width $w_n=b+I_n$. The generator checks that the energy differences and energy partial sums fit these ranges.

| $n$ | Fractional bits $b$ | Integer bits $I_n$ | Total bits $w_n$ |
|---:|---:|---:|---:|
| 8 | 18 | 6 | 24 |
| 16 | 20 | 8 | 28 |
| 32 | 22 | 9 | 31 |
| 64 | 24 | 11 | 35 |

`coeff_t` and `energy_t` share this signed format. `poly_t` uses 48 signed bits, including six integer bits; `prob_t` uses 32 unsigned bits, including one integer bit. At $\beta=4$, the same discretization budget requires two additional fractional bits.

`metropolis_probability` returns one for $\Delta H\leq0$. For positive differences below the cutoff, it evaluates a degree-12 Chebyshev approximation of $e^{-\beta_{\rm ref}\Delta H}$ by the Clenshaw recurrence and clips the result to $[0,1]$. With the per-evaluation parameter $\eta_{\rm acc}=10^{-4}$ (called `EPS_OP` in the code), the cutoff is $E_{\rm cut}^{(b)}=2^{-b}\lceil 2^b\ln(2/\eta_{\rm acc})/\beta_{\rm ref}\rceil$. At or above this cutoff, the output is zero, discarding an exact probability of at most $5\times10^{-5}$. This per-evaluation tolerance does not establish a global TV-distance bound or exact detailed balance.

`sum_energy` reduces $N$ terms through `wallace_sum`, followed by `prefix_add`. Its logical depth is $O(\log N+\log w_n)$. This describes the arithmetic reduction; the complete circuit's timing also depends on selection circuitry, synthesis, and routing.

<a id="local-move"></a>
### Local move

`local_spin_flip_operation` uses a one-hot `flip_mask` to select spin $i$ and computes $\Delta_i H^{(b)}=2x_i(h_i^{(b)}+\sum_{j\ne i}J_{ij}^{(b)}x_j)$. It reduces one field term and $n$ coupling slots, including a zero self-coupling slot. The reduction requires $O(nw_n)$ bits; coefficient storage and selection are additional resources.

The inputs are `h`, the packed upper-triangular coupling array `J`, the current `spins`, and `flip_mask`. The outputs are the **signed** `delta_energy` and approximate `accept_prob`. The kernel does not modify the configuration.

<a id="uniform-move"></a>
### Uniform move

`uniform_move_operation` computes the complete candidate energy $H^{(b)}(y)=-\sum_i h_i^{(b)}y_i-\sum_{i<j}J_{ij}^{(b)}y_i y_j$ and subtracts the supplied cached current energy. Its reduction contains $M=n(n+1)/2$ terms and requires $O(n^2w_n)$ bits.

The inputs are `h`, `J`, `proposed_spins`, and `old_energy`, which must be the energy of the current configuration under the same rounded Hamiltonian. The outputs are `new_energy`, the **signed** `delta_energy`, and `accept_prob`. Candidate generation, acceptance sampling, and state and energy-cache updates belong to the caller.

<a id="reproducibility"></a>
## Reproducibility

### How to install AMD Vitis/Vivado 2025.2

Skip this subsection if the tools are already installed. Download the **2025.2 Linux Self Extracting Web Installer** from the [official AMD download page](https://www.amd.com/en/support/downloads/adaptive-socs-and-fpgas/development-tools/2025-2.html), signing in with an AMD account when prompted. Save the downloaded file in `/scratch/mincudin/installers/amd/`.

The example below extracts the installer into `/scratch/mincudin/installers/amd/extracted` and installs the tools under `/scratch/mincudin/xlnx`. Adjust these paths for another machine. The Bash code cells contain terminal commands: copy them into a Linux terminal without the `%%bash` line, since configuration, authentication, and editing require interactive input.

Make the downloaded installer executable and extract it:

In [1]:
%%bash
set -e
mkdir -p /scratch/mincudin/installers/amd /scratch/mincudin/xlnx
cd /scratch/mincudin/installers/amd
chmod +x FPGAs_AdaptiveSoCs_Unified_SDI_2025.2_1114_2157_Lin64.bin
./FPGAs_AdaptiveSoCs_Unified_SDI_2025.2_1114_2157_Lin64.bin \
    --keep --noexec --target /scratch/mincudin/installers/amd/extracted

mkdir: cannot create directory ‘/scratch’: Permission denied
mkdir: cannot create directory ‘/scratch’: Permission denied


CalledProcessError: Command 'b'set -e\nmkdir -p /scratch/mincudin/installers/amd /scratch/mincudin/xlnx\ncd /scratch/mincudin/installers/amd\nchmod +x FPGAs_AdaptiveSoCs_Unified_SDI_2025.2_1114_2157_Lin64.bin\n./FPGAs_AdaptiveSoCs_Unified_SDI_2025.2_1114_2157_Lin64.bin \\\n    --keep --noexec --target /scratch/mincudin/installers/amd/extracted\n'' returned non-zero exit status 1.

Generate the installation configuration. Select the full **Vitis** product, which includes Vivado and Vitis HLS, and save the configuration as `~/.Xilinx/install_config.txt`. See AMD's [installation guide](https://docs.amd.com/r/2025.2-English/ug1742-vitis-release-notes/Installing-the-Vitis-Software-Platform) and [configuration instructions](https://docs.amd.com/r/2025.2-English/ug973-vivado-release-notes-install-license/Generate-a-Configuration-File).

In [ ]:
%%bash
cd /scratch/mincudin/installers/amd/extracted
./xsetup -b ConfigGen

Edit the generated configuration to set the installation destination and retain support for Virtex UltraScale+ FPGAs. Keep the other required tool selections in the existing `Modules` entry.

In [ ]:
%%bash
# Set Destination=/scratch/mincudin/xlnx.
# Ensure the Modules entry includes Virtex UltraScale+ FPGAs:1.
nano "$HOME/.Xilinx/install_config.txt"

Generate an [authentication token](https://docs.amd.com/r/2025.2-English/ug973-vivado-release-notes-install-license/Acquire-Authentication-Token) by entering your AMD account credentials when prompted, then start the [batch installation](https://docs.amd.com/r/2025.2-English/ug973-vivado-release-notes-install-license/Web-Installer-Download-and-Install). The installation command accepts the listed license agreements.

In [ ]:
%%bash
set -e
cd /scratch/mincudin/installers/amd/extracted
./xsetup -b AuthTokenGen
./xsetup --agree XilinxEULA,3rdPartyEULA \
    --batch Install --config "$HOME/.Xilinx/install_config.txt"

On Linux, an administrator can install the required system libraries with AMD's dependency helper:

In [ ]:
%%bash
sudo /scratch/mincudin/xlnx/2025.2/Vitis/scripts/installLibs.sh

Source the installed environment and check that the executables are available. Temporarily disabling `nounset` avoids errors from unset variables referenced by the AMD setup script. The workflow's `run.sh` sources this environment itself; update its hardcoded `VITIS_SETTINGS` path if you use a different installation directory. A suitable license, a C++17 compiler, and Python 3 are also required.

In [ ]:
%%bash
set -e
set +u
source /scratch/mincudin/xlnx/2025.2/Vitis/settings64.sh
set -u
command -v vitis-run
command -v vivado
vivado -version

### Run the FPGA workflow

Run the following code cell on a machine with AMD Vitis/Vivado 2025.2, support for the target FPGA, a suitable license, a C++17 compiler, and Python 3. Adjust `FPGA_DIR` to the package location and, if necessary, edit the hardcoded `VITIS_SETTINGS` path in `run.sh`.

`run.sh` takes no parameters. It uses $\beta_{\rm ref}=1$, polynomial degree 12, error parameters $10^{-4}$, test size 10, synthesis sizes $n\in\{8,16,32,64\}$, the `xcvu19p-fsva3824-2-e` device, and a target clock period of $3\,\mathrm{ns}$. It generates the headers, compiles and runs the standalone Wallace testbench, runs both move C simulations at all configured sizes, synthesizes both kernels, copies the reports, and extracts the latencies. Output is printed to the terminal, and generated Vitis logs and journals are removed on exit.

`tb_wallace.cpp` tests the compressor tree and prefix adder without Vitis headers. `tb_local.cpp` and `tb_uniform.cpp` check rounded-grid energies and acceptance probabilities, including zero and cutoff boundaries. At $n=10$, the local test covers every configuration and single-spin flip; the uniform test covers every current configuration and ten candidates per configuration. Larger sizes use sampled configurations.

In [2]:
from pathlib import Path
import subprocess

FPGA_DIR = Path("../estimation_timing/fpga").resolve()  # Adjust if extracted elsewhere.

subprocess.run(["python3", "-m", "pip", "install", "numpy", "scipy"], check=True)
subprocess.run(["bash", "run.sh"], cwd=FPGA_DIR, check=True)

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/mnt/c/Users/massi/Documents/progetti/penny_venv/lib/python3.14/site-packages/pip/__main__.py", line 24, in <module>
    sys.exit(_main())
             ~~~~~^^
  File "/mnt/c/Users/massi/Documents/progetti/penny_venv/lib/python3.14/site-packages/pip/_internal/cli/main.py", line 83, in main
    command = create_command(cmd_name, isolated=("--isolated" in cmd_args))
  File "/mnt/c/Users/massi/Documents/progetti/penny_venv/lib/python3.14/site-packages/pip/_internal/commands/__init__.py", line 121, in create_command
    module = importlib.import_module(module_path)
  File "/usr/lib/python3.14/importlib/__init__.py", line 88, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1406, in _gcd_import
  File "<fro

KeyboardInterrupt: 

<a id="timing-calculation"></a>
## Timing calculation

`reports.py` copies the synthesis reports and reads only `Worst-caseRealTimeLatency` from each top-level XML report. It converts the reported units to seconds and writes `reports/summary.csv` with the columns `move`, `n`, and `latency_s`. The values are already absolute latencies and must not be multiplied by the clock period again.

These are nominal worst-case HLS arithmetic-kernel estimates at the target clock, rather than measured complete-chain transition times. Placement and routing and the caller's proposal, random comparison, and state-update loop are outside these estimates. Hardware resources must fit the target device.

The code below reads the results generated by the preceding cell. No timings or resource counts from the previous implementation are reused.

In [3]:
import csv

with (FPGA_DIR / "reports/summary.csv").open(newline="") as report:
    rows = list(csv.DictReader(report))

print(f"{'move':8s} {'n':>4s} {'latency_s':>14s}")
for row in rows:
    print(f"{row['move']:8s} {int(row['n']):4d} {float(row['latency_s']):14.6g}")

move        n      latency_s
local       8       2.91e-07
uniform     8       2.97e-07
local      16       2.94e-07
uniform    16       3.09e-07
local      32          3e-07
uniform    32       3.27e-07
local      64       3.09e-07
uniform    64       3.87e-07


For the paper's empirical model $\tau_{\rm FPGA}(n)=a+b\log_2 n$, the following cell fits the newly extracted latencies separately for each proposal. This analysis is separate from report extraction: `reports.py` performs no fitting. The fitted model is valid only over the range supported by the synthesized data and available hardware.

In [4]:
import numpy as np

for move in ("local", "uniform"):
    data = sorted((int(r["n"]), float(r["latency_s"])) for r in rows if r["move"] == move)
    n, latency = np.array(data).T
    design = np.column_stack((np.ones_like(n), np.log2(n)))
    a, b = np.linalg.lstsq(design, latency, rcond=None)[0]
    print(f"{move}: tau(n) = {a:.6e} + {b:.6e} log2(n) s")

local: tau(n) = 2.715000e-07 + 6.000000e-09 log2(n) s
uniform: tau(n) = 2.004000e-07 + 2.880000e-08 log2(n) s
